# EDA de `merged_datasets`
Exploraremos el dataset combinado paso a paso. El objetivo es entender:
- la distribución de tipos (`Type`)
- los grupos de características `f_`, `t_`, `w_`
- qué variables parecen separar mejor los eventos sísmicos

A medida que avancemos, plantearé preguntas para definir el siguiente análisis.

In [ ]:
# Cargar librerías y datos
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.0)

path = os.path.join(r'c:\Users\ricar\OneDrive\Desktop\TESIS', 'merged_datasets.json')
with open(path, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print('shape:', df.shape)
print('columnas:', len(df.columns))
print('tipos de datos:\n', df.dtypes.value_counts())

In [ ]:
# Resumen de la etiqueta Type y grupos de características
print('Conteo de eventos por Type:')
counts = df['Type'].value_counts()
print(counts)

prefixes = {
    'f_': [c for c in df.columns if c.startswith('f_')],
    't_': [c for c in df.columns if c.startswith('t_')],
    'w_': [c for c in df.columns if c.startswith('w_')]
}
for prefix, cols in prefixes.items():
    print(f"{prefix}: {len(cols)} columnas")

plt.figure(figsize=(10, 5))
counts.plot(kind='bar')
plt.title('Distribución de eventos por Type')
plt.ylabel('Nº de registros')
plt.xlabel('Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas básicas de grupos de características
summary = []
for prefix, cols in prefixes.items():
    subset = df[cols].select_dtypes(include=[np.number])
    summary.append({
        'grupo': prefix,
        'n_features': subset.shape[1],
        'mean_of_means': subset.mean().mean(),
        'mean_of_stds': subset.std(ddof=0).mean(),
        'max_variance': subset.var(ddof=0).max(),
        'min_variance': subset.var(ddof=0).min()
    })

summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
# Buscar características más variables y diferenciales entre LP y VT
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
variances = df[numeric_cols].var(ddof=0).sort_values(ascending=False)
print('Top 10 características por varianza:')
print(variances.head(10))

# Comparar medias de los 10 features más variables entre LP y VT
main_types = ['LP', 'VT']
if set(main_types).issubset(df['Type'].unique()):
    diff_df = df[df['Type'].isin(main_types)].groupby('Type')[variances.head(10).index].mean().T
    display(diff_df)
    diff_df.plot(kind='bar', figsize=(12, 6))
    plt.title('Media de los 10 features más variables: LP vs VT')
    plt.ylabel('Media')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# PCA simple para visualizar si hay separación por tipo
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

subset = df[numeric_cols].fillna(0)
scaler = StandardScaler()
X = scaler.fit_transform(subset)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Type'] = df['Type'].values

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Type', alpha=0.6, legend='full', palette='tab10')
plt.title('PCA 2D de features numéricos')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Varianza explicada por PC1 y PC2:', pca.explained_variance_ratio_[:2])

## Direcciones interesantes para profundizar
- Puedo enfocarme en una discriminación clara entre `LP` y `VT`, que son las clases dominantes.
- También podemos investigar clases raras como `ICEQUAKE`, `TRBA` o `REGIONAL` para ver si se separan en el espacio de características.
- Otra posibilidad es cuantificar qué grupos (`f_`, `t_`, `w_`) aportan más al modelo.

### Preguntas para seguir juntos
1. ¿Te interesa más entender la separación entre los grandes grupos de eventos (`LP`/`VT`) o encontrar patrones en las clases raras?
2. ¿Quieres que use un modelo simple de clasificación para identificar las características más importantes?
3. ¿Prefieres que profundicemos en la interpretación de las características físicas (`f_`, `t_`, `w_`) o en una visualización con PCA/UMAP?